# 🍋 PyTorch para Vitrine dos Achados - Jakson Rodrigues
### De Montes Claros para o mundo - 100% prático

Notebook baseado em `Learning PyTorch with Examples` da Meta.
Dataset real: espremedor USB R$61,99 + produtos da sua loja.

**Como usar no iPhone:** Abra no colab.research.google.com > GPU ligada

## Módulo 1: Tensors (10 min) - Produto vira número

In [ ]:
import torch

# Seu espremedor como tensor: [preço, nota, vendas_mes]
espremedor = torch.tensor([61.99, 4.8, 152.0])
print(f"Espremedor USB: {espremedor}")

# Mais produtos da Vitrine dos Achados
produtos = torch.tensor([
    [61.99, 4.8, 152.0],  # Espremedor USB - Vende bem
    [89.90, 4.6, 98.0],   # Organizador - Vende bem
    [120.00, 3.2, 15.0],  # Produto caro nota baixa
    [45.50, 4.9, 210.0],  # Produto barato nota alta
    [199.90, 3.0, 5.0],   # Produto encalhado
])
print(f"\nBatch de {produtos.shape[0]} produtos:")
print(produtos)

# GPU - onde a Meta treina Llama 4
print(f"\nGPU disponível? {torch.cuda.is_available()}")
if torch.cuda.is_available():
    produtos = produtos.to('cuda')
    print("Movido para GPU!")

## Módulo 2: Autograd (10 min) - IA aprende sozinha

In [ ]:
# Simula lucro = preço * vendas * taxa
preco = torch.tensor([61.99], requires_grad=True)
vendas = torch.tensor([152.0], requires_grad=True)

lucro = preco * vendas * 0.15  # 15% comissão afiliado
lucro.backward()

print(f"Lucro: R${lucro.item():.2f}")
print(f"Se aumentar preço em R$1, lucro muda: R${preco.grad.item():.2f}")
print(f"Se vender 1 a mais, lucro muda: R${vendas.grad.item():.2f}")
print("\nÉ isso que o Llama faz bilhões de vezes para aprender!")

## Módulo 3: Criando modelo que prevê vendas (12 min)

In [ ]:
import torch.nn as nn

class PrevisorVitrine(nn.Module):
    def __init__(self):
        super().__init__()
        self.rede = nn.Sequential(
            nn.Linear(3, 16),  # 3 entradas -> 16 neurônios
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, 1),
            nn.Sigmoid()  # saída 0 a 1 = chance de vender
        )
    def forward(self, x):
        return self.rede(x)

modelo = PrevisorVitrine()
print(modelo)

# Testa com espremedor
teste = torch.tensor([[61.99, 4.8, 152.0]])
print(f"\nChance espremedor vender bem: {modelo(teste).item():.1%}")

## Módulo 4: Treinando com seus dados (15 min)

In [ ]:
# Dataset Vitrine: [preço, nota, vendas] -> vendeu_bem (1=sim, 0=não)
X = torch.tensor([
    [61.99, 4.8, 152.0],
    [89.90, 4.6, 98.0],
    [45.50, 4.9, 210.0],
    [39.99, 4.7, 180.0],
    [75.00, 4.5, 110.0],
    [120.00, 3.2, 15.0],
    [199.90, 3.0, 5.0],
    [150.00, 2.8, 8.0],
], dtype=torch.float32)

y = torch.tensor([[1.],[1.],[1.],[1.],[1.],[0.],[0.],[0.]], dtype=torch.float32)

# Normaliza preço/vendas pra treinar melhor
X_norm = X.clone()
X_norm[:,0] = X_norm[:,0] / 200.0  # preço
X_norm[:,2] = X_norm[:,2] / 250.0  # vendas

modelo = PrevisorVitrine()
criterio = nn.BCELoss()
otimizador = torch.optim.Adam(modelo.parameters(), lr=0.05)

print("Treinando...")
for epoch in range(200):
    pred = modelo(X_norm)
    loss = criterio(pred, y)
    
    otimizador.zero_grad()
    loss.backward()
    otimizador.step()
    
    if epoch % 40 == 0:
        print(f"Época {epoch}: Erro {loss.item():.3f}")

print("\nTreinado! Testando:")
with torch.no_grad():
    for i, prod in enumerate(X):
        p = modelo(X_norm[i].unsqueeze(0)).item()
        print(f"Produto {i+1} R${prod[0]:.2f} -> {p:.0%} chance")

## Módulo 5: Salvar e usar na sua Landing (13 min)

In [ ]:
# Salva modelo
torch.save(modelo.state_dict(), "modelo_vitrine.pth")
print("Modelo salvo: modelo_vitrine.pth")

# Função pra usar em novo produto da Shopee
def vai_vender_bem(preco, nota, vendas_estimadas):
    modelo.eval()
    entrada = torch.tensor([[preco/200.0, nota, vendas_estimadas/250.0]], dtype=torch.float32)
    with torch.no_grad():
        chance = modelo(entrada).item()
    return chance

# Testa novo produto que você quer afiliar
novo = vai_vender_bem(59.90, 4.8, 120)
print(f"\nNovo produto R$59,90 nota 4.8 -> {novo:.0%} de vender bem")

if novo > 0.7:
    print("✅ RECOMENDO afiliar e criar landing!")
else:
    print("❌ Não recomendo")

print("\nPróximo passo: plugar essa função na sua landing page do GitHub Pages")